many video smoothing

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pyproj import Transformer

# ============================================================
# CONFIG
# ============================================================

INPUT_DIR  = r"E:\04.Pak_ival\Dataset\sumur_bandung\4nov25\1.360_raw\need_coarse2"
OUTPUT_DIR = r"E:\04.Pak_ival\Dataset\sumur_bandung\4nov25\1.360_raw\smooth_coarse2"

N_ENDPOINT = 10   # averaging titik awal & akhir
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# LOAD ALL CSV (SORTED)
# ============================================================

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))

dfs = []
lengths = []

for path in csv_files:
    df = pd.read_csv(path)
    df = df.sort_values("t_rel_sec").reset_index(drop=True)
    dfs.append(df)
    lengths.append(len(df))

# ============================================================
# CONCAT ALL VIDEOS
# ============================================================

df_all = pd.concat(dfs, ignore_index=True)

lat = df_all["lat"].values
lon = df_all["lon"].values
alt = df_all["alt"].values
n   = len(df_all)

# ============================================================
# WGS84 → ECEF
# ============================================================

to_ecef   = Transformer.from_crs("EPSG:4979", "EPSG:4978", always_xy=True)
from_ecef = Transformer.from_crs("EPSG:4978", "EPSG:4979", always_xy=True)

X, Y, Z = to_ecef.transform(lon, lat, alt)

# ============================================================
# GLOBAL STRAIGHT 3D LINE (ECEF)
# ============================================================

# A = average start of FIRST video
A = np.array([
    X[:N_ENDPOINT].mean(),
    Y[:N_ENDPOINT].mean(),
    Z[:N_ENDPOINT].mean(),
])

# B = average end of LAST video
B = np.array([
    X[-N_ENDPOINT:].mean(),
    Y[-N_ENDPOINT:].mean(),
    Z[-N_ENDPOINT:].mean(),
])

alpha = np.linspace(0.0, 1.0, n)

X_s = A[0] + alpha * (B[0] - A[0])
Y_s = A[1] + alpha * (B[1] - A[1])
Z_s = A[2] + alpha * (B[2] - A[2])

# ============================================================
# ECEF → WGS84
# ============================================================

lon_s, lat_s, alt_s = from_ecef.transform(X_s, Y_s, Z_s)

df_all["lat"] = lat_s
df_all["lon"] = lon_s
df_all["alt"] = alt_s

# ============================================================
# SPLIT BACK TO PER-VIDEO OUTPUT
# ============================================================

offset = 0
for path, length in zip(csv_files, lengths):
    name = os.path.basename(path)
    out_path = os.path.join(OUTPUT_DIR, name)

    df_part = df_all.iloc[offset:offset + length].copy()
    offset += length

    df_part.to_csv(out_path, index=False)
    print(f"Saved → {out_path}")

print("DONE")
print("Global straight 3D trajectory applied across all videos")


Saved → E:\04.Pak_ival\Dataset\sumur_bandung\4nov25\1.360_raw\smooth_coarse2\GS013981.csv
DONE
Global straight 3D trajectory applied across all videos


: 

1 video smoothing

In [ ]:
import os
import numpy as np
import pandas as pd
from pyproj import Transformer

# ============================================================
# CONFIG
# ============================================================

INPUT_CSV  = r"D:\metadata\360\GAGAL\1.360_raw\output_smoothing\GS014244.csv"
OUTPUT_CSV = r"D:\metadata\360\GAGAL\1.360_raw\output_physical\GS014244.csv"

N_ENDPOINT = 10   # jumlah titik awal & akhir untuk averaging

# ============================================================
# LOAD CSV
# ============================================================

df = pd.read_csv(INPUT_CSV)
df = df.sort_values("t_rel_sec").reset_index(drop=True)

lat = df["lat"].values
lon = df["lon"].values
alt = df["alt"].values
n   = len(df)

# ============================================================
# WGS84 → ECEF (METRIC SPACE)
# ============================================================

to_ecef   = Transformer.from_crs("EPSG:4979", "EPSG:4978", always_xy=True)
from_ecef = Transformer.from_crs("EPSG:4978", "EPSG:4979", always_xy=True)

X, Y, Z = to_ecef.transform(lon, lat, alt)

# ============================================================
# COMPUTE STRAIGHT LINE (ECEF) — XYZ
# ============================================================

# Average first N points
A = np.array([
    X[:N_ENDPOINT].mean(),
    Y[:N_ENDPOINT].mean(),
    Z[:N_ENDPOINT].mean(),
])

# Average last N points
B = np.array([
    X[-N_ENDPOINT:].mean(),
    Y[-N_ENDPOINT:].mean(),
    Z[-N_ENDPOINT:].mean(),
])

# Linear interpolation parameter
alpha = np.linspace(0.0, 1.0, n)

X_s = A[0] + alpha * (B[0] - A[0])
Y_s = A[1] + alpha * (B[1] - A[1])
Z_s = A[2] + alpha * (B[2] - A[2])

# ============================================================
# ECEF → WGS84
# ============================================================

lon_s, lat_s, alt_s = from_ecef.transform(X_s, Y_s, Z_s)

# ============================================================
# SAVE OUTPUT
# ============================================================

df_out = df.copy()
df_out["lat"] = lat_s
df_out["lon"] = lon_s
df_out["alt"] = alt_s

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df_out.to_csv(OUTPUT_CSV, index=False)

print("DONE")
print("Trajectory forced to straight 3D line (XYZ, ECEF-based)")
print(f"Saved → {OUTPUT_CSV}")
